# 1.0 Dataset Introduction

The Kepler space telescope was launched by NASA in 2009 to find planets orbiting stars outside our solar system (exoplanets). It found them using the transit method: it continuously measured the brightness of around 150,000 stars and recorded the small, regular dips in light that occur when a planet passes in front of its star. Each repeating dip worth investigating is logged as a Kepler Object of Interest (KOI).

Not every KOI is a planet. Similar dips can be caused by two stars eclipsing each other, by a nearby star contaminating the measurement, or by instrument noise. Each KOI is therefore reviewed and given a disposition: confirmed planet, candidate still awaiting confirmation, or false positive. Doing this by hand is slow, which makes the dataset well suited to machine learning. A model can learn, from the measured properties of the signal and its host star, to separate real planets from false positives.

# 1.1 Dataset Identification

The dataset was retrieved from Kaggle, where it was uploaded under NASA's account as "Kepler Exoplanet Search Results". It is a copy of the Kepler Objects of Interest (KOI) cumulative table, which is published by the NASA Exoplanet Archive (NExScI-Caltech/IPAC, doi:10.26133/NEA4). The table brings together the results of NASA's searches of light curves from the Kepler space telescope. Although the copy was obtained from Kaggle, the data originate from a primary scientific archive and can be checked against it. The dataset can be found at the following URL: 

https://www.kaggle.com/datasets/nasa/kepler-exoplanet-search-results

The dataset contains fifty (50) attributes over 9,564 rows, where each row is one candidate planet. The target is koi_disposition. Four attributes are identifiers and are excluded from modelling, which leaves forty-five (45) features. The attributes are defined as the following types:

- <mark>Categorical (Nominal)</mark>: koi_disposition (CONFIRMED, CANDIDATE, FALSE POSITIVE), koi_pdisposition (CANDIDATE, FALSE POSITIVE), koi_tce_delivname (one of three data releases).
- <mark>Categorical (Binary)</mark>: koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, koi_fpflag_ec These are stored as 0/1 flags, each marking a specific reason the signal may be a false positive.
- <mark>Identifiers</mark>: rowid, kepid, kepoi_name, kepler_name These label the object and carry no predictive meaning. kepler_name is only filled for confirmed planets, so 76% of its values are missing.
- <mark>Numerical (Discrete)</mark>: koi_tce_plnt_num The planet's number within its star system, an integer between 1 and 8.
- <mark>Numerical (Continuous)</mark>: koi_score (0 to 1), koi_period, koi_impact, koi_duration, koi_depth, koi_prad, koi_teq, koi_insol, koi_model_snr, koi_steff, koi_slogg, koi_srad, ra, dec and koi_kepmag. Most of these come with an upper (_err1) and lower (_err2) uncertainty, which are also continuous.
- <mark>Time</mark>: koi_time0bk The time of the first observed transit. It is stored as a decimal number of days in Barycentric Kepler Julian Date (BKJD) rather than as a date string, for example 170.53875.
koi_teq_err1 and koi_teq_err2 are entirely empty and koi_score is missing for 16% of rows. koi_score, koi_pdisposition and the four false-positive flags are closely linked to the target, so they should be excluded when classifying koi_disposition. Table 2 summarises the attributes.

| Attribute(s) | Data type | Variable type | Description |
|---|---|---|---|
| rowid, kepid, kepoi_name, kepler_name | Integer / String | Identifier | Row, star and planet identifiers |
| koi_disposition | String | Categorical (nominal) | Target: archive disposition of the candidate |
| koi_pdisposition | String | Categorical (nominal) | Disposition using Kepler data alone |
| koi_tce_delivname | String | Categorical (nominal) | Data release the detection came from |
| koi_fpflag_nt / ss / co / ec | Integer | Categorical (binary) | False-positive flags: not transit-like, stellar eclipse, centroid offset, ephemeris match |
| koi_tce_plnt_num | Float | Numerical (discrete) | Planet number within the system |
| koi_time0bk | Float | Time | Transit epoch (BKJD days) |
| koi_score | Float | Numerical (continuous) | Confidence in the disposition, 0 to 1 |
| koi_period, koi_duration, koi_depth, koi_impact | Float | Numerical (continuous) | Orbital period (days), transit duration (hours), transit depth (ppm), impact parameter |
| koi_prad, koi_teq, koi_insol | Float | Numerical (continuous) | Planet radius (Earth radii), equilibrium temperature (K), insolation flux |
| koi_model_snr | Float | Numerical (continuous) | Transit signal-to-noise ratio |
| koi_steff, koi_slogg, koi_srad, koi_kepmag | Float | Numerical (continuous) | Star temperature (K), surface gravity, radius (Solar radii), Kepler magnitude |
| ra, dec | Float | Numerical (continuous) | Sky coordinates (degrees) |
| *_err1, *_err2 | Float | Numerical (continuous) | Upper and lower uncertainty of the matching attribute |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import accuracy_score, recall_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score